In [ ]:
%load_ext autoreload

%autoreload 2

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

# Inspect Dataset — ABO Furniture

Visual inspection of the curated ABO furniture classroom subset.  
Covers: class distribution, original resolution spread, sample grids per class, retrieval group inspection, ImageFolder loading, DataLoader batch.

## Imports

In [ ]:
import csv
import json
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pyrootutils
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

## Parameters

In [ ]:
ROOT = pyrootutils.setup_root(
    search_from=".",
    indicator="pyproject.toml",
    project_root_env_var=True,
    dotenv=True,
    pythonpath=True,
    cwd=True,
)

In [ ]:
DATA_DIR = ROOT / "data/abo"
SPLITS = ["train", "val", "test"]
SEED = 0

assert DATA_DIR.exists(), f"Not found: {DATA_DIR.resolve()}"
print(f"Dataset path: {DATA_DIR.resolve()}")

## Section 1 — Class Distribution

Count images per class and split from `metadata.csv`, then plot a bar chart.

In [ ]:
with (DATA_DIR / "metadata.csv").open() as f:
    rows = list(csv.DictReader(f))

print(f"Total images: {len(rows)}")
print(f"Columns: {list(rows[0].keys())}\n")

counts = defaultdict(Counter)  # counts[split][label]
for r in rows:
    counts[r["split"]][r["label"]] += 1

all_labels = sorted({r["label"] for r in rows})
col_w = 8
print(
    f"{'label':<12}" + "".join(f"{s:>{col_w}}" for s in SPLITS) + f"{'total':>{col_w}}"
)
print("-" * (12 + col_w * (len(SPLITS) + 1)))
for label in all_labels:
    cs = [counts[s][label] for s in SPLITS]
    print(f"{label:<12}" + "".join(f"{c:>{col_w}}" for c in cs) + f"{sum(cs):>{col_w}}")
totals = [sum(counts[s].values()) for s in SPLITS]
print(
    f"{'TOTAL':<12}"
    + "".join(f"{t:>{col_w}}" for t in totals)
    + f"{sum(totals):>{col_w}}"
)

In [ ]:
train_counts = [counts["train"][label] for label in all_labels]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(all_labels, train_counts, color="steelblue", edgecolor="white")
_ = ax.bar_label(bars, fmt="%d", padding=3)
_ = ax.set_title("ABO furniture classroom — train images per class", fontsize=13)
_ = ax.set_ylabel("Image count")
_ = ax.set_xlabel("Class")
plt.tight_layout()
plt.show()

max_c, min_c = max(train_counts), min(train_counts)
print(f"Max/min ratio (train): {max_c}/{min_c} = {max_c / min_c:.1f}x")

## Section 2 — Views per Item

Each product item has a primary image plus up to 3 additional views. Inspect how many images come from primary vs non-primary views, and the distribution of views per item.

In [ ]:
primary_count = sum(1 for r in rows if r["is_primary"] == "True")
print(f"Primary images:     {primary_count:>6}")
print(f"Non-primary images: {len(rows) - primary_count:>6}")
print(f"Total:              {len(rows):>6}")

# View index distribution
view_counter = Counter(int(r["view_index"]) for r in rows)
print("\nImages per view_index:")
for vi in sorted(view_counter):
    print(f"  view_index={vi}: {view_counter[vi]:>6}")

In [ ]:
# Images per item
item_images = defaultdict(list)
for r in rows:
    item_images[r["item_id"]].append(r)

views_per_item = [len(v) for v in item_images.values()]
view_hist = Counter(views_per_item)

print(f"Total items:         {len(item_images)}")
print(f"Avg images / item:   {sum(views_per_item) / len(views_per_item):.2f}")
print(f"Min / max per item:  {min(views_per_item)} / {max(views_per_item)}")
print("\nDistribution of images per item:")
for n_views in sorted(view_hist):
    bar = "█" * (view_hist[n_views] * 40 // max(view_hist.values()))
    print(f"  {n_views:>2} views: {view_hist[n_views]:>5} items  {bar}")

## Section 3 — Original Resolution Distribution

Images were centre-cropped to 224×224 before saving. Inspect the original source resolutions to understand the input diversity.

In [ ]:
orig_w = [int(r["original_width"]) for r in rows]
orig_h = [int(r["original_height"]) for r in rows]

print(
    f"Original width  — min: {min(orig_w)}, max: {max(orig_w)}, median: {sorted(orig_w)[len(orig_w) // 2]}"
)
print(
    f"Original height — min: {min(orig_h)}, max: {max(orig_h)}, median: {sorted(orig_h)[len(orig_h) // 2]}"
)

# Square images?
n_square = sum(1 for w, h in zip(orig_w, orig_h, strict=False) if w == h)
print(f"\nSquare images: {n_square}/{len(rows)} ({100 * n_square / len(rows):.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
_ = sns.histplot(orig_w, bins=40, color="steelblue", ax=axes[0])
_ = axes[0].set_title("Original width")
_ = axes[0].set_xlabel("Pixels")
_ = sns.histplot(orig_h, bins=40, color="coral", ax=axes[1])
_ = axes[1].set_title("Original height")
_ = axes[1].set_xlabel("Pixels")
plt.suptitle(
    "ABO furniture — original image resolution (before 224x224 crop)", fontsize=12
)
plt.tight_layout()
plt.show()

## Section 4 — Sample Grid per Class

4 × 4 random training samples per class. Check for: wrong labels, poor crops, obvious artefacts.

In [ ]:
random.seed(SEED)

train_index = defaultdict(list)
for label_dir in sorted((DATA_DIR / "train").iterdir()):
    if label_dir.is_dir():
        train_index[label_dir.name] = sorted(label_dir.iterdir())

N_COLS, N_ROWS = 4, 4
for label in all_labels:
    paths = train_index.get(label, [])
    sample = random.sample(paths, min(N_COLS * N_ROWS, len(paths)))
    n = len(sample)
    n_rows = (n + N_COLS - 1) // N_COLS

    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 2.5, n_rows * 2.5))
    fig.suptitle(f"class: {label}  (n_train={len(paths)})", fontsize=12, y=1.01)
    axes_flat = axes.flatten() if n > 1 else [axes]

    for ax, p in zip(axes_flat, sample, strict=False):
        with Image.open(p) as im:
            _ = ax.imshow(im)
        ax.axis("off")
    for ax in axes_flat[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Section 5 — Retrieval Group Inspection

`retrieval_groups.json` groups all views of the same product under its `item_id`. Show all views of a few selected items — these form the ground-truth relevance sets used in retrieval evaluation.

In [ ]:
with (DATA_DIR / "retrieval_groups.json").open() as f:
    retrieval_groups = json.load(f)

total_groups = len(retrieval_groups)
multi_view = sum(1 for g in retrieval_groups.values() if len(g["image_ids"]) >= 2)
print(f"Total groups (items):       {total_groups}")
print(
    f"Groups with ≥2 views:       {multi_view} ({100 * multi_view / total_groups:.1f}%)"
)

sizes = Counter(len(g["image_ids"]) for g in retrieval_groups.values())
print("\nViews per group:")
for n in sorted(sizes):
    bar = "█" * (sizes[n] * 40 // max(sizes.values()))
    print(f"  {n} views: {sizes[n]:>5} groups  {bar}")

In [ ]:
# Build a quick lookup: image_id -> file_path
id_to_path = {r["image_id"]: DATA_DIR / r["file_path"] for r in rows}

# Show one multi-view item per class (pick the item with the most views)
random.seed(SEED)

for label in all_labels:
    label_groups = {
        iid: g
        for iid, g in retrieval_groups.items()
        if g["label"] == label and g["split"] == "train" and len(g["image_ids"]) >= 2
    }
    if not label_groups:
        print(f"{label}: no multi-view train items found")
        continue

    best_item = max(label_groups, key=lambda iid: len(label_groups[iid]["image_ids"]))
    image_ids = label_groups[best_item]["image_ids"]

    n = len(image_ids)
    fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 3))
    fig.suptitle(f"class: {label} — item {best_item} ({n} views)", fontsize=11)
    axes_flat = axes if n > 1 else [axes]

    for ax, img_id in zip(axes_flat, image_ids, strict=False):
        with Image.open(id_to_path[img_id]) as im:
            _ = ax.imshow(im)
        _ = ax.set_title(img_id[:12], fontsize=7)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Section 6 — ImageFolder Loading

Verify that torchvision's `ImageFolder` reads the dataset correctly and that class indices are consistent.

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_ds = ImageFolder(DATA_DIR / "train", transform=transform)
val_ds = ImageFolder(DATA_DIR / "val", transform=transform)
test_ds = ImageFolder(DATA_DIR / "test", transform=transform)

print(f"Classes:    {train_ds.classes}")
print(f"Train size: {len(train_ds)}")
print(f"Val size:   {len(val_ds)}")
print(f"Test size:  {len(test_ds)}")

img, label = train_ds[0]
print(f"\nSample tensor — shape: {img.shape}, dtype: {img.dtype}")
print(f"Value range:  [{img.min():.2f}, {img.max():.2f}]  (normalised, not [0,1])")

## Section 7 — DataLoader Batch

Load one batch and display it as a grid (after un-normalising for visualisation).

In [ ]:
loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)

imgs, labels = next(iter(loader))
print(f"Batch shape:  {imgs.shape}")
print(f"Labels:       {[train_ds.classes[label_idx] for label_idx in labels.tolist()]}")

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
imgs_disp = (imgs * std + mean).clamp(0, 1)

grid = make_grid(imgs_disp, nrow=8, padding=4)
fig, ax = plt.subplots(figsize=(18, 5))
_ = ax.imshow(grid.permute(1, 2, 0).numpy())
_ = ax.set_title(
    "ABO furniture — one training batch (224x224, displayed after un-normalising)",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## Section 8 — Sanity Checks

Verify that the number of files on disk matches the manifest, and that no image files are corrupt.

In [ ]:
# File count vs manifest
disk_files = list(DATA_DIR.rglob("*.jpg"))
print(f"Files on disk:    {len(disk_files)}")
print(f"Manifest entries: {len(rows)}")
assert len(disk_files) == len(rows), "Mismatch between disk and manifest!"
print("OK — counts match")

In [ ]:
# Corrupt image detection — try opening every file
corrupt = []
for p in disk_files:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        corrupt.append((p, str(e)))

print(f"Corrupt files: {len(corrupt)}")
for p, err in corrupt:
    print(f"  {p}: {err}")
assert len(corrupt) == 0, f"{len(corrupt)} corrupt files found"
print("OK — all images are readable")